In [1]:
# Imports
import time
import librosa
import numpy as np
import torch
import jiwer
import pandas as pd
from whisper.normalizers.basic import BasicTextNormalizer
from tqdm.notebook import tqdm
from transformers import VoxtralRealtimeProcessor, VoxtralRealtimeForConditionalGeneration

# Dispositivo = GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Ejecutando en: {device}")

Ejecutando en: cuda:0


In [2]:
# Cargar modelo y procesador
repo_id = "mistralai/Voxtral-Mini-4B-Realtime-2602"
print(f"Cargando modelo y procesador: {repo_id}")

processor = VoxtralRealtimeProcessor.from_pretrained(repo_id)
model = VoxtralRealtimeForConditionalGeneration.from_pretrained(repo_id).to(device)
model.eval()

audio_config = processor.mistral_common_audio_config
sr = processor.feature_extractor.sampling_rate
hop = processor.feature_extractor.hop_length
win = processor.feature_extractor.win_length

ms_por_token = processor.audio_length_per_tok * hop / sr * 1000
ms_por_chunk = processor.num_samples_per_audio_chunk / sr * 1000

print(f"Retardo por defecto: {audio_config.transcription_delay_ms} ms ({processor.num_delay_tokens} tokens)")
print(f"Chunk de streaming: {ms_por_chunk} ms de audio")

Cargando modelo y procesador: mistralai/Voxtral-Mini-4B-Realtime-2602


Loading weights:   0%|          | 0/711 [00:00<?, ?it/s]

Retardo por defecto: 480 ms (6 tokens)
Chunk de streaming: 105.0 ms de audio


In [3]:
N_AUDIOS = 20
RETARDOS_MS = [160, 320, 480, 960, 1920]

base = "../../Europarl-ST/es/en/test"
lst_path = base + "/Europarl-ST.v2.es.en.test.lst"

audios = []
with open(lst_path, "r", encoding="utf-8") as lista_audios:
    for linea in lista_audios:
        if linea.strip():
            audios.append(str(linea).strip())

audios = audios[:N_AUDIOS]
print(f"Audios: {len(audios)} | configuraciones: {RETARDOS_MS}")

Audios: 20 | configuraciones: [160, 320, 480, 960, 1920]


In [4]:
def trocear(audio_array):
    entrada = processor(
        audio_array[:processor.num_samples_first_audio_chunk],
        is_streaming=True,
        is_first_audio_chunk=True,
        return_tensors="pt",
    ).to(model.device, dtype=model.dtype)

    trozos = [entrada.input_features]
    mel = processor.num_mel_frames_first_audio_chunk
    start = mel * hop - win // 2

    while (end := start + processor.num_samples_per_audio_chunk) < audio_array.shape[0]:
        t = processor(
            audio_array[start:end],
            is_streaming=True,
            is_first_audio_chunk=False,
            return_tensors="pt",
        )
        trozos.append(t.to(model.device, dtype=model.dtype).input_features)
        mel += processor.audio_length_per_tok
        start = mel * hop - win // 2

    return entrada, trozos


def transcribir_streaming(audio_array):
    entrada, trozos = trocear(audio_array)
    tiempos = []

    def generador():
        for t in trozos:
            inicio = time.perf_counter()
            yield t
            torch.cuda.synchronize()
            tiempos.append((time.perf_counter() - inicio) * 1000)

    t0 = time.perf_counter()
    with torch.no_grad():
        salida = model.generate(
            input_ids=entrada.input_ids,
            input_features=generador(),
            num_delay_tokens=entrada.num_delay_tokens,
        )
    total = time.perf_counter() - t0

    texto = processor.batch_decode(salida, skip_special_tokens=True)[0]
    return texto, np.array(tiempos[1:]), total

In [5]:
filas = []

for retardo in RETARDOS_MS:
    audio_config.transcription_delay_ms = retardo

    for audio in tqdm(audios, desc=f"retardo {retardo} ms"):

        ref_path = base + "/%s/transcription.tok" % audio
        with open(ref_path, "r", encoding="utf-8") as referencia:
            ref = referencia.read()

        audio_path = base + "/%s/audio_clip_diarization.m4a" % audio
        audio_array, _ = librosa.load(audio_path, sr=sr)
        npad = processor.num_right_pad_tokens
        npad = npad() if callable(npad) else npad
        audio_array = np.pad(audio_array, (0, npad * processor.raw_audio_length_per_tok))

        hyp, por_chunk, total = transcribir_streaming(audio_array)
        duracion = len(audio_array) / sr

        filas.append(dict(
            audio=audio,
            retardo_ms=retardo,
            latencia_computacional_ms=float(por_chunk.mean()),
            latencia_p90_ms=float(np.percentile(por_chunk, 90)),
            latencia_primer_tercio_ms=float(por_chunk[:len(por_chunk) // 3].mean()),
            latencia_ultimo_tercio_ms=float(por_chunk[-len(por_chunk) // 3:].mean()),
            duracion_audio_s=duracion,
            tiempo_total_s=total,
            rtf=total / duracion,
            hypothesis=hyp,
            reference=ref,
        ))

audio_config.transcription_delay_ms = 480

data = pd.DataFrame(filas)
data.to_csv("voxtral_latencia_raw_es.csv", index=False)
data.head()

retardo 160 ms:   0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_1869/329943581.py:13: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(audio_path, sr=sr)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/transformers/generation/utils.py:1805: UserWarning: Using the model-agnostic default `max_length` (=1000000000) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/tmp/ipykernel_1869/329943581.py:13: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(audio_path, sr=sr)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio._

retardo 320 ms:   0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_1869/329943581.py:13: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(audio_path, sr=sr)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/transformers/generation/utils.py:1805: UserWarning: Using the model-agnostic default `max_length` (=1000000000) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/tmp/ipykernel_1869/329943581.py:13: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(audio_path, sr=sr)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio._

retardo 480 ms:   0%|          | 0/20 [00:00<?, ?it/s]

...

retardo 960 ms:   0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_1869/329943581.py:13: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(audio_path, sr=sr)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/transformers/generation/utils.py:1805: UserWarning: Using the model-agnostic default `max_length` (=1000000000) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/tmp/ipykernel_1869/329943581.py:13: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(audio_path, sr=sr)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio._

retardo 1920 ms:   0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_1869/329943581.py:13: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(audio_path, sr=sr)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/transformers/generation/utils.py:1805: UserWarning: Using the model-agnostic default `max_length` (=1000000000) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/tmp/ipykernel_1869/329943581.py:13: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(audio_path, sr=sr)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio._

,audio,retardo_ms,latencia_computacional_ms,latencia_p90_ms,latencia_primer_tercio_ms,latencia_ultimo_tercio_ms,duracion_audio_s,tiempo_total_s,rtf,hypothesis,reference
0,en.20081008.24.3-238,160,72.910347,97.539987,72.000481,74.522721,125.057812,118.405877,0.946809,"Gracias, señor presidente. En primer lugar, q...","−\nSeñor Presidente , en primer lugar , quisie..."
1,en.20081020.12.1-036,160,69.811579,92.054545,67.537573,73.391479,47.038750,40.862848,0.868706,El Atlético de Madrid los aficionados e inclu...,"El Atlético de Madrid , los aficionados e incl..."
2,en.20081020.15.1-139,160,69.073740,91.473823,68.589208,68.213701,63.037312,54.260167,0.860763,"Gracias, señor presidente. Señor comisario, e...","Señor Presidente , señor Comisario , el terror..."
3,en.20081118.31.2-399,160,68.948820,87.520312,68.374620,70.182316,129.028437,111.177067,0.861648,"Muchas gracias, señor presidente. Yo quiero a...","Señor Presidente , quiero agradecer a la Comis..."
4,en.20081118.4.2-040,160,69.097269,87.271013,68.875897,69.532080,131.025313,113.147630,0.863556,"Gracias, presidente. Una gran mayoría de agri...","Señor Presidente , una gran mayoría de agricul..."


In [6]:
# Normalización de texto para evaluación
normalizer = BasicTextNormalizer()

data["hypothesis_clean"] = [normalizer(str(t)) for t in data["hypothesis"]]
data["reference_clean"] = [normalizer(str(t)) for t in data["reference"]]

data.to_csv("voxtral_latencia_limpio_es.csv", index=False)
data.head()

,audio,retardo_ms,latencia_computacional_ms,latencia_p90_ms,latencia_primer_tercio_ms,latencia_ultimo_tercio_ms,duracion_audio_s,tiempo_total_s,rtf,hypothesis,reference,hypothesis_clean,reference_clean
0,en.20081008.24.3-238,160,72.910347,97.539987,72.000481,74.522721,125.057812,118.405877,0.946809,"Gracias, señor presidente. En primer lugar, q...","−\nSeñor Presidente , en primer lugar , quisie...",gracias señor presidente en primer lugar quis...,señor presidente en primer lugar quisiera fel...
1,en.20081020.12.1-036,160,69.811579,92.054545,67.537573,73.391479,47.038750,40.862848,0.868706,El Atlético de Madrid los aficionados e inclu...,"El Atlético de Madrid , los aficionados e incl...",el atlético de madrid los aficionados e inclu...,el atlético de madrid los aficionados e inclus...
2,en.20081020.15.1-139,160,69.073740,91.473823,68.589208,68.213701,63.037312,54.260167,0.860763,"Gracias, señor presidente. Señor comisario, e...","Señor Presidente , señor Comisario , el terror...",gracias señor presidente señor comisario el t...,señor presidente señor comisario el terrorismo...
3,en.20081118.31.2-399,160,68.948820,87.520312,68.374620,70.182316,129.028437,111.177067,0.861648,"Muchas gracias, señor presidente. Yo quiero a...","Señor Presidente , quiero agradecer a la Comis...",muchas gracias señor presidente yo quiero agr...,señor presidente quiero agradecer a la comisió...
4,en.20081118.4.2-040,160,69.097269,87.271013,68.875897,69.532080,131.025313,113.147630,0.863556,"Gracias, presidente. Una gran mayoría de agri...","Señor Presidente , una gran mayoría de agricul...",gracias presidente una gran mayoría de agricu...,señor presidente una gran mayoría de agriculto...


In [7]:
# WER y latencia por configuración
resumen = []

for retardo, grupo in data.groupby("retardo_ms"):
    wer = jiwer.wer(list(grupo["reference_clean"]), list(grupo["hypothesis_clean"]))
    computacional = grupo["latencia_computacional_ms"].mean()
    resumen.append(dict(
        modelo="Voxtral",
        latencia_algoritmica_ms=retardo,
        latencia_computacional_ms=computacional,
        latencia_total_ms=retardo + computacional,
        crecimiento=grupo["latencia_ultimo_tercio_ms"].mean() / grupo["latencia_primer_tercio_ms"].mean(),
        rtf=grupo["rtf"].mean(),
        wer=wer * 100,
    ))

resumen = pd.DataFrame(resumen)
resumen.to_csv("voxtral_latencia_resumen_es.csv", index=False)
resumen

,modelo,latencia_algoritmica_ms,latencia_computacional_ms,latencia_total_ms,crecimiento,rtf,wer
0,Voxtral,160,71.511284,231.511284,1.010928,0.892546,16.976404
1,Voxtral,320,70.176997,390.176997,1.026422,0.871117,13.969513
2,Voxtral,480,67.155821,547.155821,1.000090,0.831358,9.688870
3,Voxtral,960,67.970339,1027.970339,1.013812,0.833684,7.412821
4,Voxtral,1920,69.554689,1989.554689,1.028875,0.838511,19.983295


In [8]:
print(f"Cada chunk son {ms_por_chunk:.1f} ms de audio.")
print("Si la latencia computacional supera ese valor el sistema acumula retraso.")
print()

for _, f in resumen.iterrows():
    sigue = "si" if f["latencia_computacional_ms"] < ms_por_chunk else "NO"
    print(
        f"algoritmica {int(f['latencia_algoritmica_ms']):5d} ms | "
        f"computacional {f['latencia_computacional_ms']:7.1f} ms | "
        f"total {f['latencia_total_ms']:7.1f} ms | "
        f"crecimiento x{f['crecimiento']:.2f} | "
        f"WER {f['wer']:5.2f} % | tiempo real: {sigue}"
    )

Cada chunk son 105.0 ms de audio.
Si la latencia computacional supera ese valor el sistema acumula retraso.

algoritmica   160 ms | computacional    71.5 ms | total   231.5 ms | crecimiento x1.01 | WER 16.98 % | tiempo real: si
algoritmica   320 ms | computacional    70.2 ms | total   390.2 ms | crecimiento x1.03 | WER 13.97 % | tiempo real: si
algoritmica   480 ms | computacional    67.2 ms | total   547.2 ms | crecimiento x1.00 | WER  9.69 % | tiempo real: si
algoritmica   960 ms | computacional    68.0 ms | total  1028.0 ms | crecimiento x1.01 | WER  7.41 % | tiempo real: si
algoritmica  1920 ms | computacional    69.6 ms | total  1989.6 ms | crecimiento x1.03 | WER 19.98 % | tiempo real: si
